#### This is an explanation of the operations the notebook does
---
The cell bellow makes the necessary imports for the libraries that are going to be used, and establishes a connection with the data lake storage system -minio- by calling the appropriate function that already exists in the configuration file of the framework

In [307]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config

client = config.create_minio_client()

[Bucket('aws'), Bucket('azure'), Bucket('google')]


### Step 0: Data Ingestion from MinIO

In the cell below, the MinIO object storage is accessed through the client created in the previous step. A specific raw JSON object representing a Google Cloud service is downloaded and stored in the corresponding variable. 

Afterwards, using pandas, this raw JSON is transformed into an initial DataFrame. At this starting point, the DataFrame contains only two top-level key-value pairs:
1. **`skus`**: A nested list containing all the individual services/products (`[list of services]`).
2. **`nextPageToken`**: A string token used for API pagination (`"String"`).

In [ ]:
"""
Kubernetes_Engine: 1 page, 1.4 Mib, 1731rows, produces 26 columns
Compute_Engine: 7 pages, 4.1Mib and 1.1Mib, 5000 rows/page, produces 29 columns
Cloud_Storage: 1 page, 1000Kib, 1220 services, 3000 with duplicates, produces 29 columns
Cloud_SQL : 4 pages, produces 29 columns
Networking: 1 page, 1600 services, 2800 non unique, produces 29 columns

"""
object_name = "Compute_Engine/page_1.json"

try:
    response = client.get_object(config.PROVIDERS.get("google").get("bucket"), object_name=object_name)

    df = pd.read_json(response)

    response.close()
    response.release_conn()

    print ("Success. Page loaded and converted into dataframe")
    print ("Array size: Rows = ",df.shape[0], " and Columns = ", df.shape[1])

except Exception as e:
    print ("Error: ",e)


Success. Page loaded and converted into dataframe
Array size: Rows =  1421  and Columns =  2


### Step 1: Loading & Initial Flattening

In the cell below, the first key-value pair is flattened and the list of services is opened, with each service occupying a row. Also, any first-level inner dictionaries (dicts) that a service may contain are also opened.

For example, a nested structure like this:
`category: {key1: value1, key2: value2}`

Opens up and flattens into distinct columns:
`category.key1`, `category.key2` with their respective values mapped across the rows.

In [309]:
#With the commnand bellow, we open the first level key : value pairs in columns, and the first level inner dicts also open, 
#in the form key.value (ex: category.serviceDisplayName, <- This was an inner dict category :{key:value, key:value})

df_flat = pd.json_normalize(df['skus'])
df_flat.head(3)
# df_flat[['skuId', 'category.serviceDisplayName', 'category.resourceFamily', 'category.usageType', 'category.resourceGroup']].head()

,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/F3FA-6DBB-9418,F3FA-6DBB-9418,Sole Tenancy Premium for C4 Sole Tenancy Insta...,[asia-east2],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,[asia-east2]
1,services/6F81-5844-456A/skus/F400-A443-F5C2,F400-A443-F5C2,DWS Defined Duration N1 Predefined Ram running...,[europe-southwest1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,[europe-southwest1]
2,services/6F81-5844-456A/skus/F402-0D26-40C8,F402-0D26-40C8,Nvidia Tesla A100 GPU attached to Spot Preempt...,[us-west3],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,GPU,Preemptible,REGIONAL,[us-west3]


### Step 2: Exploding Geographic Regions

In the cell below, we perform an `.explode()` operation on the `geoTaxonomy.regions` column, which originally contains a nested list of locations. 

By exploding this list, each region occupies its own row. This means that for services available in multiple locations, duplicated rows are created for all other attributes, with the only difference being the specific region value in each row.

In [310]:
#Here geoTaxonomy.regions looks like this: ["value"]
df_flat[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()

#We apply the explode in the geoTaxonomy.regions column and store the res in a new dataframe
df_flat2 = df_flat.explode('geoTaxonomy.regions')

#The result will be: geoTaxonomy wont be a list anymore, and all the list elements will be in a single line
# df_flat2[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()
df_flat2.head(3)


,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/F3FA-6DBB-9418,F3FA-6DBB-9418,Sole Tenancy Premium for C4 Sole Tenancy Insta...,[asia-east2],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,asia-east2
1,services/6F81-5844-456A/skus/F400-A443-F5C2,F400-A443-F5C2,DWS Defined Duration N1 Predefined Ram running...,[europe-southwest1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-southwest1
2,services/6F81-5844-456A/skus/F402-0D26-40C8,F402-0D26-40C8,Nvidia Tesla A100 GPU attached to Spot Preempt...,[us-west3],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,GPU,Preemptible,REGIONAL,us-west3


### Step 3: Exploding Service Regions

In the cell below, we perform the exact same `.explode()` operation on the service regions column. 

Just like with the geotaxonomy regions, this operation unpacks the nested list of service regions into individual rows, resulting in duplicated rows.

In [311]:
#We apply the same proccedure as the cell above. This time we open the list serviceRegions
df_flat2[['skuId', 'serviceRegions']].head()

df_flat3 = df_flat2.explode('serviceRegions')

df_flat3[['skuId', 'serviceRegions', 'geoTaxonomy.regions']].head()

df_flat3.head()

,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/F3FA-6DBB-9418,F3FA-6DBB-9418,Sole Tenancy Premium for C4 Sole Tenancy Insta...,asia-east2,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,asia-east2
1,services/6F81-5844-456A/skus/F400-A443-F5C2,F400-A443-F5C2,DWS Defined Duration N1 Predefined Ram running...,europe-southwest1,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-southwest1
2,services/6F81-5844-456A/skus/F402-0D26-40C8,F402-0D26-40C8,Nvidia Tesla A100 GPU attached to Spot Preempt...,us-west3,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,GPU,Preemptible,REGIONAL,us-west3
3,services/6F81-5844-456A/skus/F402-F22E-E665,F402-F22E-E665,Spot Preemptible Custom Extended Instance Ram ...,me-central1,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,RAM,Preemptible,REGIONAL,me-central1
4,services/6F81-5844-456A/skus/F405-06FA-BB84,F405-06FA-BB84,Spot Preemptible Z3 Instance Local SSD running...,europe-west10,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,LocalSSD,Preemptible,REGIONAL,europe-west10


### Step 4: Exploding Pricing information

Just like the 2 cells above an `.explode()` operation is made to unpack the list of `pricingInfo`, and isolate each of the elements in a single row. For the record, the value within the `pricingInfo` list is a dictionary, which contains the detailed pricing structures and rates for each service

In [312]:
#The only column not fully opened yet is the pricingInfo: structure -> [{key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}}]
df_flat3[['skuId', 'pricingInfo']].head()

#This first explode removes the list: we have now -> {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} 
df_flat4 = df_flat3.explode('pricingInfo')
df_flat4[['skuId', 'pricingInfo']].head()
df_flat4.head()

,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/F3FA-6DBB-9418,F3FA-6DBB-9418,Sole Tenancy Premium for C4 Sole Tenancy Insta...,asia-east2,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,asia-east2
1,services/6F81-5844-456A/skus/F400-A443-F5C2,F400-A443-F5C2,DWS Defined Duration N1 Predefined Ram running...,europe-southwest1,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-southwest1
2,services/6F81-5844-456A/skus/F402-0D26-40C8,F402-0D26-40C8,Nvidia Tesla A100 GPU attached to Spot Preempt...,us-west3,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,GPU,Preemptible,REGIONAL,us-west3
3,services/6F81-5844-456A/skus/F402-F22E-E665,F402-F22E-E665,Spot Preemptible Custom Extended Instance Ram ...,me-central1,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,RAM,Preemptible,REGIONAL,me-central1
4,services/6F81-5844-456A/skus/F405-06FA-BB84,F405-06FA-BB84,Spot Preemptible Z3 Instance Local SSD running...,europe-west10,"{'summary': '', 'pricingExpression': {'usageUn...",Google,Compute Engine,Compute,LocalSSD,Preemptible,REGIONAL,europe-west10


### Step 5: Flattening the pricingInfo Dictionary

At this stage of the pipeline, the pricingInfo column contains a dict with the following structure:
`{key1: val1, key2: val2, key3: val3, pricingExpression: {key: val, tieredRates: [{}]}}`

To extract these nested properties into standalone columns, we use the `pd.json_normalize()` again.

**What this operation does:**
1. **Unpacks the Dictionary:** It takes the keys of the `pricingInfo` dictionary (such as `effectiveTime`, `summary`, `currencyConversionRate` and `pricingExpression`) and turns them into separate, clean columns.
2. **Handles Sub-Nested Structures:** If a key contains further nested dictionaries (like `pricingExpression`), it automatically flattens them using dot notation (e.g., `pricingExpression.pricingUnits`, `pricingExpression.baseUnit`).
3. **Preserves Sub-Lists:** Deeply nested lists (like `pricingExpression.tieredRates` which contains the actual price tiers) are kept intact inside their cells, preparing them for the final cleaning steps.

In [313]:
#We are here now: {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} -> we can open the dict with the normalize
#A new dataframe will be created with the pricing info and then concatenated with the original dataframe

pricing1 = pd.json_normalize(df_flat4['pricingInfo'])
pricing1.head()

,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.tieredRates,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount
0,,1,2026-06-07T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
1,,1,2026-06-07T07:00:00Z,GiBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN
2,,1,2026-06-07T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
3,,1,2026-06-07T07:00:00Z,GiBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN
4,,1,2026-06-07T07:00:00Z,GiBy.mo,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gibibyte month,By.s,byte second,2.783139e+15,NaN,NaN,NaN


### Step 6: Aligning Indexes and Merging Data

Since `normalize()` creates a new DataFrame (`pricing1` in our case) we have  to merge it back with our main DataFrame to reconstruct the complete dataset. During this concatenation it is imperative to align the indexes of each line, and drop the pricingInfo column from the first dataset, since it's data will now be in standalone cols


In [314]:
#I now have to concatenate the 2 dataframes beeing carefull though with the indexes
pricing1.index = df_flat4.index

df_flat5 = pd.concat([df_flat4.drop(columns=['pricingInfo']), pricing1], axis=1)
df_flat5.head()
# df_flat5['pricingExpression.tieredRates']

,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.tieredRates,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount
0,services/6F81-5844-456A/skus/F3FA-6DBB-9418,F3FA-6DBB-9418,Sole Tenancy Premium for C4 Sole Tenancy Insta...,asia-east2,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,asia-east2,,1,2026-06-07T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
1,services/6F81-5844-456A/skus/F400-A443-F5C2,F400-A443-F5C2,DWS Defined Duration N1 Predefined Ram running...,europe-southwest1,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-southwest1,,1,2026-06-07T07:00:00Z,GiBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN
2,services/6F81-5844-456A/skus/F402-0D26-40C8,F402-0D26-40C8,Nvidia Tesla A100 GPU attached to Spot Preempt...,us-west3,Google,Compute Engine,Compute,GPU,Preemptible,REGIONAL,us-west3,,1,2026-06-07T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
3,services/6F81-5844-456A/skus/F402-F22E-E665,F402-F22E-E665,Spot Preemptible Custom Extended Instance Ram ...,me-central1,Google,Compute Engine,Compute,RAM,Preemptible,REGIONAL,me-central1,,1,2026-06-07T07:00:00Z,GiBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN
4,services/6F81-5844-456A/skus/F405-06FA-BB84,F405-06FA-BB84,Spot Preemptible Z3 Instance Local SSD running...,europe-west10,Google,Compute Engine,Compute,LocalSSD,Preemptible,REGIONAL,europe-west10,,1,2026-06-07T07:00:00Z,GiBy.mo,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gibibyte month,By.s,byte second,2.783139e+15,NaN,NaN,NaN


### Step 7: Exploding Pricing Tiers

In the cell below, we perform the final `.explode()` operation on the `pricingExpression.tieredRates` column. 
Since a single SKU can have multiple pricing tiers, this operation unpacks the nested list of tiers into individual rows, ensuring every distinct price rate is isolated for analysis.
The value of the list was dictionary/ries so the next operation that is requires is, flattening that dictionary and then concatenation. Those operations take place in the 3 cells bellow -the detailed procedure is not explained as is simillar with above-

In [315]:
df_flat6 = df_flat5.explode('pricingExpression.tieredRates')

df_flat6[['skuId', 'pricingExpression.tieredRates']].head()

,skuId,pricingExpression.tieredRates
0,F3FA-6DBB-9418,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
1,F400-A443-F5C2,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
2,F402-0D26-40C8,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
3,F402-F22E-E665,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
4,F405-06FA-BB84,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."


In [316]:
pricing2 = pd.json_normalize(df_flat6['pricingExpression.tieredRates'])
pricing2.head()

,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos
0,0.0,USD,0,4847188.0
1,0.0,USD,0,4999660.0
2,0.0,USD,1,713200000.0
3,0.0,USD,0,1100000.0
4,0.0,USD,0,12900000.0


In [317]:
pricing2.index = df_flat6.index

df_final_flat = pd.concat([df_flat6.drop(columns=['pricingExpression.tieredRates']), pricing2], axis=1)
pd.set_option('display.max_columns', None)
df_final_flat.head()

,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos
0,services/6F81-5844-456A/skus/F3FA-6DBB-9418,F3FA-6DBB-9418,Sole Tenancy Premium for C4 Sole Tenancy Insta...,asia-east2,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,asia-east2,,1,2026-06-07T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,4847188.0
1,services/6F81-5844-456A/skus/F400-A443-F5C2,F400-A443-F5C2,DWS Defined Duration N1 Predefined Ram running...,europe-southwest1,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-southwest1,,1,2026-06-07T07:00:00Z,GiBy.h,1,gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN,0.0,USD,0,4999660.0
2,services/6F81-5844-456A/skus/F402-0D26-40C8,F402-0D26-40C8,Nvidia Tesla A100 GPU attached to Spot Preempt...,us-west3,Google,Compute Engine,Compute,GPU,Preemptible,REGIONAL,us-west3,,1,2026-06-07T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,1,713200000.0
3,services/6F81-5844-456A/skus/F402-F22E-E665,F402-F22E-E665,Spot Preemptible Custom Extended Instance Ram ...,me-central1,Google,Compute Engine,Compute,RAM,Preemptible,REGIONAL,me-central1,,1,2026-06-07T07:00:00Z,GiBy.h,1,gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN,0.0,USD,0,1100000.0
4,services/6F81-5844-456A/skus/F405-06FA-BB84,F405-06FA-BB84,Spot Preemptible Z3 Instance Local SSD running...,europe-west10,Google,Compute Engine,Compute,LocalSSD,Preemptible,REGIONAL,europe-west10,,1,2026-06-07T07:00:00Z,GiBy.mo,1,gibibyte month,By.s,byte second,2.783139e+15,NaN,NaN,NaN,0.0,USD,0,12900000.0


### Step 8: Calculating the Final Price (Handling Units & Nanos)

According to Google Cloud's official documentation, SKU prices are not represented as standard floats. Instead, they are split into two separate components to prevent floating-point precision errors:
* `units`: The whole number part of the price.
* `nanos`: The fractional part of the price, represented as billionths of a cent/dollar ($10^{-9}$).

For example, a price of **$1.75** is stored as `units = 1` and `nanos = 750,000,000`.

**In the cell below, we:**: reconstruct the price by converting both columns to floats, divide `unitPrice.nanos` by $1,000,000,000$, and add them together to create the clean `finalPrice` column.


In [318]:
#The cost of the SKU is units + nanos. For example, a cost of $1.75 is represented as units=1 and nanos=750,000,000. (From google documentation)
#Actions need to be made to create a new column that will contain the final price
df_final_flat['finalPrice'] = df_final_flat['unitPrice.units'].astype(float) + (df_final_flat['unitPrice.nanos'].astype(float) / 1000000000)

df_final_flat[['skuId', 'unitPrice.units', 'unitPrice.nanos', 'finalPrice']]

,skuId,unitPrice.units,unitPrice.nanos,finalPrice
0,F3FA-6DBB-9418,0,4847188.0,0.004847
1,F400-A443-F5C2,0,4999660.0,0.005000
2,F402-0D26-40C8,1,713200000.0,1.713200
3,F402-F22E-E665,0,1100000.0,0.001100
4,F405-06FA-BB84,0,12900000.0,0.012900
...,...,...,...,...
1418,FFFE-B3F4-43FB,0,39678780.0,0.039679
1419,FFFF-B27D-95FA,0,150000000.0,0.150000
1419,FFFF-B27D-95FA,0,130000000.0,0.130000
1419,FFFF-B27D-95FA,0,110000000.0,0.110000


In [319]:
#Just for debug to search if any line has "1" as unit price
filtered_df = df_final_flat[df_final_flat['unitPrice.units'].astype(float) == 1.0]

filtered_df[['skuId','unitPrice.units', 'unitPrice.nanos', 'finalPrice']].head()

# df_final_flat.head()

,skuId,unitPrice.units,unitPrice.nanos,finalPrice
2,F402-0D26-40C8,1,713200000.0,1.713200
30,F434-024D-6602,1,86592000.0,1.086592
119,F501-6A5E-AC7F,1,129555000.0,1.129555
171,F578-1482-10F0,1,220000000.0,1.220000
269,F650-9ADC-2EB9,1,460000000.0,1.460000


In [320]:
print(df_final_flat['unitPrice.currencyCode'].value_counts())
print ()
print(df_final_flat.shape[0])
print()
print(df_final_flat['skuId'].nunique())

unitPrice.currencyCode
USD    2134
Name: count, dtype: int64

2136

1421


In [321]:
#Just fot debug: filtering lines where the currencny is not USD
eur_rows_df = df_final_flat[df_final_flat['unitPrice.currencyCode'] != 'USD']

print(df_final_flat['unitPrice.currencyCode'].value_counts(dropna=False))

eur_rows_df.head()

unitPrice.currencyCode
USD    2134
NaN       2
Name: count, dtype: int64


,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos,finalPrice
398,services/6F81-5844-456A/skus/F75A-F7F2-B1D6,F75A-F7F2-B1D6,Cloud Interconnect - Data Transfer North Ameri...,global,Google,Compute Engine,Network,PeeringOrInterconnectEgress,OnDemand,GLOBAL,NaN,Sku is not being priced by default.,0,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
401,services/6F81-5844-456A/skus/F75E-CD53-1A28,F75E-CD53-1A28,Network Internet Data Transfer Out from Seoul ...,asia-northeast3,Google,Compute Engine,Network,PremiumInternetEgress,OnDemand,REGIONAL,asia-northeast3,Sku is not being priced by default.,0,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 9: Cleaning, Currency normalization, Duplicate removal

**Operations Performed:**
1. We remove any rows where `unitPrice.currencyCode` is absent, filtering out unpriced or incomplete SKU records.

2. We convert all prices into (USD) by dividing the `finalPrice` by the provided `currencyConversionRate`. Easier for future analytics

3. For data profiling purposes we create a separate subset that keeps only the first occurrence of each unique `skuId`. This ensures accurate statistical metrics.


In [324]:
#Throw duplicates, throw records with Nan in currencycode, new column with usd final price

df_clean_no_nan = df_final_flat.dropna(subset=['unitPrice.currencyCode']) #If the value is nan in this column drop (If the product has no currency registered its problematic)

#Currency normalization
rate = df_clean_no_nan['currencyConversionRate'].astype(float)

#This is our clean array so far
df_clean_no_nan['final_price_usd'] = df_clean_no_nan['finalPrice'].astype(float) / rate

#Only for profiling tool drop all duplicates
df_for_profiling = df_clean_no_nan.drop_duplicates(subset=['skuId'])

print(f"The size of the array is {df_clean_no_nan.shape[0]} rows, and {df_clean_no_nan.shape[1]}, columns")
print (f"The size of the array used for profiling is {df_for_profiling.shape[0]} rows and {df_for_profiling.shape[1]} columns")
df_clean_no_nan.head()
# print(df_for_profiling.shape[0])

The size of the array is 2134 rows, and 29, columns
The size of the array used for profiling is 1419 rows and 29 columns


/tmp/ipykernel_223734/171766326.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean_no_nan['final_price_usd'] = df_clean_no_nan['finalPrice'].astype(float) / rate


,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos,finalPrice,final_price_usd
0,services/6F81-5844-456A/skus/F3FA-6DBB-9418,F3FA-6DBB-9418,Sole Tenancy Premium for C4 Sole Tenancy Insta...,asia-east2,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,asia-east2,,1,2026-06-07T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,4847188.0,0.004847,0.004847
1,services/6F81-5844-456A/skus/F400-A443-F5C2,F400-A443-F5C2,DWS Defined Duration N1 Predefined Ram running...,europe-southwest1,Google,Compute Engine,Compute,N1Standard,OnDemand,REGIONAL,europe-southwest1,,1,2026-06-07T07:00:00Z,GiBy.h,1,gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN,0.0,USD,0,4999660.0,0.005000,0.005000
2,services/6F81-5844-456A/skus/F402-0D26-40C8,F402-0D26-40C8,Nvidia Tesla A100 GPU attached to Spot Preempt...,us-west3,Google,Compute Engine,Compute,GPU,Preemptible,REGIONAL,us-west3,,1,2026-06-07T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,1,713200000.0,1.713200,1.713200
3,services/6F81-5844-456A/skus/F402-F22E-E665,F402-F22E-E665,Spot Preemptible Custom Extended Instance Ram ...,me-central1,Google,Compute Engine,Compute,RAM,Preemptible,REGIONAL,me-central1,,1,2026-06-07T07:00:00Z,GiBy.h,1,gibibyte hour,By.s,byte second,3.865471e+12,NaN,NaN,NaN,0.0,USD,0,1100000.0,0.001100,0.001100
4,services/6F81-5844-456A/skus/F405-06FA-BB84,F405-06FA-BB84,Spot Preemptible Z3 Instance Local SSD running...,europe-west10,Google,Compute Engine,Compute,LocalSSD,Preemptible,REGIONAL,europe-west10,,1,2026-06-07T07:00:00Z,GiBy.mo,1,gibibyte month,By.s,byte second,2.783139e+15,NaN,NaN,NaN,0.0,USD,0,12900000.0,0.012900,0.012900


In [323]:
# from ydata_profiling import ProfileReport

# profile = ProfileReport(df_for_profiling, title="Google Cloud Compute Engine - Unique SKUs Report", explorative=True)

# #Stores in file under the same directory
# profile.to_file("google_billing_unique_analysis.html")

# print("Report created")

**Debug purposes**

In [ ]:
#To check witch of the aggreagation columns actually have a value. Most of them dont
df_clean_no_nan[df_clean_no_nan['aggregationInfo.aggregationCount'].notna()]

,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos,finalPrice,final_price_usd
28,services/6F81-5844-456A/skus/F42B-53B3-C4A0,F42B-53B3-C4A0,Network Vpn Internet Data Transfer Out from Ja...,asia-northeast1,Google,Compute Engine,Network,VPNInternetEgress,OnDemand,REGIONAL,asia-northeast1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,0.0,USD,0,140000000.0,0.14,0.14
28,services/6F81-5844-456A/skus/F42B-53B3-C4A0,F42B-53B3-C4A0,Network Vpn Internet Data Transfer Out from Ja...,asia-northeast1,Google,Compute Engine,Network,VPNInternetEgress,OnDemand,REGIONAL,asia-northeast1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,10240.0,USD,0,120000000.0,0.12,0.12
62,services/6F81-5844-456A/skus/F47A-B6D4-25D2,F47A-B6D4-25D2,Network Vpn Internet Data Transfer Out from Fi...,europe-north1,Google,Compute Engine,Network,VPNInternetEgress,OnDemand,REGIONAL,europe-north1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,0.0,USD,0,120000000.0,0.12,0.12
62,services/6F81-5844-456A/skus/F47A-B6D4-25D2,F47A-B6D4-25D2,Network Vpn Internet Data Transfer Out from Fi...,europe-north1,Google,Compute Engine,Network,VPNInternetEgress,OnDemand,REGIONAL,europe-north1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,1024.0,USD,0,110000000.0,0.11,0.11
62,services/6F81-5844-456A/skus/F47A-B6D4-25D2,F47A-B6D4-25D2,Network Vpn Internet Data Transfer Out from Fi...,europe-north1,Google,Compute Engine,Network,VPNInternetEgress,OnDemand,REGIONAL,europe-north1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,10240.0,USD,0,80000000.0,0.08,0.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1414,services/6F81-5844-456A/skus/FFEE-F021-9F25,FFEE-F021-9F25,Network Internet Data Transfer Out from Israel...,me-west1,Google,Compute Engine,Network,PremiumInternetEgress,OnDemand,REGIONAL,me-west1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,1024.0,USD,0,180000000.0,0.18,0.18
1414,services/6F81-5844-456A/skus/FFEE-F021-9F25,FFEE-F021-9F25,Network Internet Data Transfer Out from Israel...,me-west1,Google,Compute Engine,Network,PremiumInternetEgress,OnDemand,REGIONAL,me-west1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,10240.0,USD,0,150000000.0,0.15,0.15
1419,services/6F81-5844-456A/skus/FFFF-B27D-95FA,FFFF-B27D-95FA,Network Internet Data Transfer Out from APAC t...,asia-east1,Google,Compute Engine,Network,PremiumInternetEgress,OnDemand,REGIONAL,asia-east1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,0.0,USD,0,150000000.0,0.15,0.15
1419,services/6F81-5844-456A/skus/FFFF-B27D-95FA,FFFF-B27D-95FA,Network Internet Data Transfer Out from APAC t...,asia-east1,Google,Compute Engine,Network,PremiumInternetEgress,OnDemand,REGIONAL,asia-east1,,1,2026-06-07T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,ACCOUNT,MONTHLY,1.0,1024.0,USD,0,130000000.0,0.13,0.13
